# Create RDS files for Seurat Label Transfer

**Pinned Environment:** [`envs/R_integration.yaml`](../.../envs/R_integration.yaml)  

In [ ]:
library(Seurat)

In [18]:
repo_root <- normalizePath(file.path(getwd(), ".."))

cmd <- paste0(
  "PYTHONPATH=", repo_root,
  " python3 -c 'import sys; from pathlib import Path; sys.path.append(str(Path.cwd().resolve().parents[1])); ",
  "from config.paths import BASE_DIR; print(BASE_DIR)'"
)

base_dir <- file.path(system(cmd, intern = TRUE), "data", "rds")

adata_dir   <- file.path(base_dir, "xenium")
refdata_dir <- file.path(base_dir, "refdata")

In [ ]:
load_seurat_from_csv <- function(dataset_dir, project_name) {

  message("Loading dataset from: ", dataset_dir)

  # Determine assay name
  assay_name <- if (grepl("xenium", project_name, ignore.case = TRUE)) "Xenium" else "RNA"

  # Load counts
  zip_file <- file.path(dataset_dir, "counts_matrix.zip")
  stopifnot(file.exists(zip_file))

  tmpdir <- tempfile()
  dir.create(tmpdir)
  unzip(zip_file, exdir = tmpdir)

  mtx_file      <- file.path(tmpdir, "matrix.mtx.gz")
  barcodes_file <- file.path(tmpdir, "barcodes.tsv.gz")
  features_file <- file.path(tmpdir, "features.tsv.gz")
  cell_md_file  <- file.path(dataset_dir, "metadata", "cell-metadata.csv")

  stopifnot(file.exists(mtx_file))
  stopifnot(file.exists(barcodes_file))
  stopifnot(file.exists(features_file))
  stopifnot(file.exists(cell_md_file))

  counts_matrix <- ReadMtx(
    mtx            = mtx_file,
    cells          = barcodes_file,
    features       = features_file,
    feature.column = 1
  )

  # Build Seurat object
  seu <- CreateSeuratObject(counts = counts_matrix, project = project_name)

  # Rename assay for Xenium
  if (assay_name == "Xenium") {
    seu <- RenameAssays(seu, RNA = "Xenium")
  }

  # Load cell metadata
  md <- read.csv(cell_md_file)
  rownames(md) <- md[,1]
  md <- md[,-1, drop = FALSE]
  md <- md[colnames(seu), , drop = FALSE]
  seu <- AddMetaData(seu, md)

  # Load feature metadata
  feature_metadata_file <- file.path(dataset_dir, "metadata", "feature-metadata.csv")
  stopifnot(file.exists(feature_metadata_file))

  feature_metadata <- read.csv(feature_metadata_file)
  rownames(feature_metadata) <- feature_metadata[,1]

  common_genes <- intersect(rownames(seu), rownames(feature_metadata))
  feature_metadata <- feature_metadata[common_genes, , drop = FALSE]

  for (colname in colnames(feature_metadata)) {
    seu[[colname]] <- feature_metadata[[colname]]
  }

  # Load dimensional reductions
  reduc_dir <- file.path(dataset_dir, "reductions")

  latent_file <- file.path(reduc_dir, "latent_representation.csv")
  umap_file   <- file.path(reduc_dir, "umap.csv")

  stopifnot(file.exists(latent_file))
  stopifnot(file.exists(umap_file))

  latent <- read.csv(latent_file, row.names = 1)
  colnames(latent) <- paste0("latent_", seq_len(ncol(latent)))
  seu[["latent"]] <- CreateDimReducObject(as.matrix(latent), key = "latent_", assay = assay_name)

  umap <- read.csv(umap_file, row.names = 1)
  colnames(umap) <- paste0("umap_", seq_len(ncol(umap)))
  seu[["umap"]] <- CreateDimReducObject(as.matrix(umap), key = "UMAP_", assay = assay_name)

  return(seu)
}

In [ ]:
adata_seurat   <- load_seurat_from_csv(adata_dir,   project_name = "adata_xenium")
refdata_seurat <- load_seurat_from_csv(refdata_dir, project_name = "refdata_seq")

In [ ]:
# Save both .RDS outputs
save_dir <- file.path(base_dir, "data", "rds", "seurat")
dir.create(save_dir, recursive = TRUE, showWarnings = FALSE)

saveRDS(adata_seurat,   file = file.path(save_dir, "adata-xenium.rds"))
saveRDS(refdata_seurat, file = file.path(save_dir, "refdata-seq.rds"))

message("Saved RDS files:")
message(" - ", file.path(save_dir, "adata-xenium.rds"))
message(" - ", file.path(save_dir, "refdata-seq.rds"))